# STEP 1 — 데이터 확보

> ## 🚨 먼저 읽으세요 — 이 노트북은 Colab/Kaggle 에서 다운로드가 **안 됩니다**
>
> AI Hub 는 **한국 IP 에서만 다운로드를 허용**합니다.
> Colab/Kaggle VM 은 한국 밖에 있어서 이렇게 막힙니다:
> ```
> Download failed with HTTP status 502.
> AI 허브는 해외에서의 데이터 다운로드를 제한하고 있습니다.
> ```
> 목록 조회(`-mode l`)는 되는데 **다운로드만** 막히니 헷갈리기 쉽습니다.
>
> ### 해결: 한국 PC 에서 받고 전처리까지 한 뒤, 크롭본만 올립니다
>
> ```bash
> # 내 컴퓨터(한국)에서
> git clone https://github.com/gayeoniee/deeplearning_test.git
> cd deeplearning_test && pip install -r requirements.txt
> export AIHUB_API_KEY="발급받은키"
> python prepare_local.py --all
> # → dogskin_prepared.zip (2~5GB) 생성 → Kaggle/Drive 에 비공개 업로드
> ```
>
> 원본 21GB 가 크롭 후 2~5GB 로 줄어서 업로드가 현실적입니다.
> 전처리는 **CPU 만** 쓰므로 GPU 없는 노트북에서도 됩니다.
>
> 📖 자세한 절차: [`docs/cautions/06_해외IP_다운로드_차단_우회.md`](../docs/cautions/06_해외IP_다운로드_차단_우회.md)
>
> **로컬에서 `prepare_local.py --all` 을 돌렸다면 이 노트북은 건너뛰고
> `03_학습_베이스라인.ipynb` 로 바로 가세요.**

---

아래는 **한국에 있는 서버/PC 에서 주피터로 실행**하는 경우의 절차입니다.

## 시작 전 체크리스트

1. [AI Hub](https://aihub.or.kr) 회원가입
2. [해당 데이터셋 페이지](https://aihub.or.kr/aihubdata/data/view.do?dataSetSn=561)에서 **활용신청** → 승인 대기 (보통 1영업일)
3. 마이페이지에서 **API Key 발급** (이메일로 옵니다)
4. 발급받은 키를 아래 위치에 등록

| 환경 | 등록 위치 |
|---|---|
| Colab | 왼쪽 사이드바 🔑 **보안 비밀** → 이름 `AIHUB_API_KEY` → **노트북 액세스** 토글 ON |
| Kaggle | Add-ons → **Secrets** → 이름 `AIHUB_API_KEY` → 이 노트북에 Attach |
| 로컬 | `export AIHUB_API_KEY=...` |

> ⚠️ **키를 노트북 셀에 직접 붙여넣지 마세요.**

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

## 1. aihubshell 설치

AI Hub 가 제공하는 공식 CLI 입니다. 웹에서 클릭으로 받는 것보다 훨씬 편하고,
무엇보다 **필요한 파일만 골라 받을 수 있습니다.**

In [ ]:
from src import aihub

aihub.install()
APIKEY = env.secret("AIHUB_API_KEY")   # ← 값은 화면에 찍히지 않습니다
print("API Key 로드 완료 (길이:", len(APIKEY), ")")

## 2. 파일 목록 확인

`-mode l` 로 데이터셋 안에 어떤 파일이 있는지 먼저 봅니다.

전체가 500GB 급이라 **절대 통째로 받으면 안 됩니다.** Colab 디스크는 100GB 남짓이고
세션이 끊기면 다 날아갑니다.

In [ ]:
files = aihub.list_files(APIKEY)

### 파싱이 0건이면?

AI Hub 출력 포맷이 바뀐 것입니다. 아래로 원본을 직접 보고 filekey 를 눈으로 고른 뒤,
다음 셀에서 `picks` 를 수동으로 지정하세요.

In [ ]:
# 파싱 실패했을 때만 실행
# print(aihub.raw_listing(APIKEY)[:6000])

## 3. 다운로드 전략 ★

⚠️ **파일이 6개 통짜 zip 으로만 나뉘어 있습니다.** (총 382GB)
"반려견+일반카메라만 골라 받기"는 파일 단위로 **불가능**합니다.

대신 이렇게 갑니다:

| | |
|---|---|
| **Training(340GB) 은 안 받습니다** | Colab 디스크로 불가능하고, 필요하지도 않습니다. 어차피 `split.py` 로 **개체 단위 재분할**을 하므로 AI Hub 의 Training/Validation 구분을 따를 이유가 없습니다 |
| **VL01(라벨, 21GB) 먼저** | 라벨 zip 과 원천 zip 의 크기가 **정확히 같습니다** (90=90, 80=80, 21=21). JSON 만 21GB 일 리 없으니 **이미지가 함께 들어있을 가능성**이 높습니다 |
| **확인 후 결정** | 이미지가 있으면 VS01 은 안 받아도 됩니다. JSON 만이면 VS01 을 추가로 받습니다 |

반려견/일반카메라 필터링은 **다운로드가 아니라 전처리 단계**(`labels.build()`)에서 경로 기준으로 걸립니다.

In [ ]:
aihub.recommend_plan()

## 4. 1단계 — VL01(라벨) 만 받기

21GB 입니다. 네트워크 속도에 따라 10~30분 걸립니다.
받는 도중 세션이 끊겨도 이 셀을 다시 실행하면 이어집니다.

In [ ]:
VL01 = "517022"      # Validation 라벨, 21GB
failed = aihub.download(APIKEY, [VL01], chunk=1)

In [ ]:
aihub.unpack_all()          # 자동 해제가 안 됐으면 마저 풀기
info = aihub.peek()         # ★ 안에 이미지가 함께 있는지 확인

### 결과에 따라 분기

- **`✅ 이미지와 JSON이 함께 있습니다`** → VS01 안 받아도 됩니다. **5번 건너뛰고 노트북 01로.**
- **`ℹ️ JSON만 있습니다`** → 아래 5번에서 VS01(이미지)을 추가로 받습니다.

## 5. 2단계 — VS01(이미지) 받기 *(위에서 JSON만 나온 경우에만)*

⚠️ 디스크를 확인하세요. VL01 이 이미 자리를 차지하고 있어서 빠듯할 수 있습니다.

In [ ]:
print("여유 디스크:", env.free_disk_gb(), "GB")

# VS01 = "517021"     # Validation 원천(이미지), 21GB
# failed = aihub.download(APIKEY, [VS01], chunk=1)
# aihub.unpack_all()
# aihub.peek()

## 6. Google Drive 백업 (Colab, 선택)

Colab 세션은 끊기면 `/content` 가 통째로 사라집니다.
**원본 전체를 Drive 에 올리지는 마세요** — 무료 15GB 로는 어림도 없고, Drive I/O 가 느려서
학습이 오히려 느려집니다.

권장: 원본은 세션 디스크에 두고, **STEP 3에서 만든 크롭본 + 매니페스트만** Drive 에 백업.

In [ ]:
# DRIVE = env.mount_drive()
# print(DRIVE)

---
## ✅ 다음 단계

`01_데이터_스캔_EDA.ipynb` 로 넘어가세요.

거기서 **데이터가 실제로 어떻게 생겼는지** 알아냅니다 —
JSON 키 이름, 개체ID 필드, 무증상 데이터 존재 여부, 병변 크기, 중복률.
전처리 코드를 짜기 전에 반드시 거쳐야 하는 단계입니다.

> 💡 이번에 받은 건 AI Hub 기준 "Validation" 이지만, 우리는 이걸
> **전체 데이터로 보고 `split.py` 로 개체 단위 재분할**합니다.
> AI Hub 의 원래 분할은 개체 누수를 보장하지 않기 때문입니다.

📖 함께 읽기: [`docs/cautions/01_데이터_라이선스와_재배포_금지.md`](../docs/cautions/01_데이터_라이선스와_재배포_금지.md)